In [1]:
# Import Packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
import logging
import gwpy
import sys
from gwpy.timeseries import TimeSeries

In [27]:
# Logger
logging.basicConfig(
    level=20, # set to 10 (DEBUG) for Development/ 20 (INFO) for PROD
    format="%(asctime)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
    force=True
)

In [28]:
spark = SparkSession.builder \
    .master("local") \
    .appName("Ligo_pipeline") \
    .getOrCreate()

In [3]:
detector = "H1"
gps_start = 1126259462
duration= 4096 # Sec
sample_rate = 4096

In [4]:
# Getting TimeSeries data from gwpy library
data = TimeSeries.fetch_open_data(
    detector, 
    gps_start, 
    gps_start + duration, 
    sample_rate
)

2026-05-23 15:25:35,854 - DEBUG - fetching https://gwosc.org/api/v2/event-versions?min-gps-time=1126259462&max-gps-time=1126267654
2026-05-23 15:25:35,896 - DEBUG - Starting new HTTPS connection (1): gwosc.org:443
2026-05-23 15:25:36,255 - DEBUG - https://gwosc.org:443 "GET /api/v2/event-versions?min-gps-time=1126259462&max-gps-time=1126267654 HTTP/1.1" 200 1069
2026-05-23 15:25:36,262 - DEBUG - fetching https://gwosc.org/api/v2/event-versions/GW150914-v4/strain-files?detector=H1&sample-rate=4&file-format=hdf
2026-05-23 15:25:36,274 - DEBUG - Starting new HTTPS connection (1): gwosc.org:443
2026-05-23 15:25:36,625 - DEBUG - https://gwosc.org:443 "GET /api/v2/event-versions/GW150914-v4/strain-files?detector=H1&sample-rate=4&file-format=hdf HTTP/1.1" 200 112
2026-05-23 15:25:36,634 - DEBUG - fetching https://gwosc.org/api/v2/event-versions/GW150914-v4
2026-05-23 15:25:36,647 - DEBUG - Starting new HTTPS connection (1): gwosc.org:443
2026-05-23 15:25:36,904 - DEBUG - https://gwosc.org:443

In [7]:
print(data)
print(type(data))
print(len(data))
print(data.sample_rate)
print(data.duration)
print(data.t0)

TimeSeries([5.16251157e-20, 3.72676369e-20, 2.76847613e-20, ...,
            2.65035515e-19, 2.39260773e-19, 2.42696492e-19]
           unit: dimensionless,
           t0: 1126259462.0 s,
           dt: 0.000244140625 s,
           name: Strain,
           channel: None)
<class 'gwpy.timeseries.timeseries.TimeSeries'>
16777216
4096.0 Hz
4096.0 s
1126259462.0 s


In [33]:
# Check for memory
bytes_per_sample = 8 # For LIGO strain float64 = 8 bytes
total_samples = duration * sample_rate # len(data.value)
memory = total_samples * bytes_per_sample
print(f"Memory Usage in MB: {memory/1000000}")
print(f"Memory Usage in MiB (Mebibyte): {memory/1048576}") # It uses powers of 2 --> 2^10=1024, 2^20=1,048,576

Memory Usage in MB: 134.217728
Memory Usage in MiB (Mebibyte): 128.0


In [18]:
print(type(data.value))
print(data.value.shape)
print(data.value[:5])

<class 'numpy.ndarray'>
(16777216,)
[5.16251157e-20 3.72676369e-20 2.76847613e-20 4.03078351e-20
 6.01961406e-20]


In [30]:
# Create bronze record
bronze_metadata = {
    "detector": detector,
    "gps_start": float(data.t0.value),
    "gps_end": float(data.t0.value + data.duration.value),
    "duration": float(data.duration.value),
    "sample_rate": float(data.sample_rate.value),
    "num_samples": len(data.value),
    "source": "GWOSC"
}

In [31]:
bronze_metadata

{'detector': 'H1',
 'gps_start': 1126259462.0,
 'gps_end': 1126263558.0,
 'duration': 4096.0,
 'sample_rate': 4096.0,
 'num_samples': 16777216,
 'source': 'GWOSC'}

In [32]:
bronze_metadata_df = spark.createDataFrame([bronze_metadata])
bronze_metadata_df.printSchema()
bronze_metadata_df.show(truncate=False)

root
 |-- detector: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- gps_end: double (nullable = true)
 |-- gps_start: double (nullable = true)
 |-- num_samples: long (nullable = true)
 |-- sample_rate: double (nullable = true)
 |-- source: string (nullable = true)

+--------+--------+-------------+-------------+-----------+-----------+------+
|detector|duration|gps_end      |gps_start    |num_samples|sample_rate|source|
+--------+--------+-------------+-------------+-----------+-----------+------+
|H1      |4096.0  |1.126263558E9|1.126259462E9|16777216   |4096.0     |GWOSC |
+--------+--------+-------------+-------------+-----------+-----------+------+



In [34]:
window_seconds = 2
samples_per_window = int(sample_rate * window_seconds)
samples_per_window # 16,777,216 / 8192 = 2048 rows/indows

In [35]:
samples_per_window

8192